# qobuzdl-collab — Google Colab

Download albums / tracks / full artist discographies from Qobuz.

**Features**
- Multi-token support (region switching)
- Hi-Res preferred
- Structure: `Artist / Year - Album / 01 - Title.flac`

Repo: https://github.com/zenin-373/qobuzdl-collab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zenin-373/qobuzdl-collab/blob/main/qobuzdl_collab_colab.ipynb)

## 1. Install

Run once per Colab session. Uses a fixed path so nested clones don't break anything.

In [ ]:
from pathlib import Path
import shutil

# Always work from a clean, fixed location
WORK = Path("/content/qobuzdl-collab")
if WORK.exists():
    shutil.rmtree(WORK)

!pip install -q click requests rich mutagen
!git clone -q https://github.com/zenin-373/qobuzdl-collab.git /content/qobuzdl-collab
%cd /content/qobuzdl-collab

# Download engine (not yet in repo due to size)
!curl -s -o qobuz_dl/downloader.py https://raw.githubusercontent.com/jcomicsutils/qobuz-dl/main/qobuz_dl/downloader.py

!pip install -q -e .
print("Installed at /content/qobuzdl-collab")

## 2. Config

Put your real credentials below, then run the cell.

- Config is always written to `/root/.config/qobuz-dl/config.json` (same path the CLI uses).
- For **new token versions**, use the `app_id` / `secret` shown by your token checker (example: `798273057` / `abb21364945c0583309667d13ca3d93a`).

In [ ]:
import json
from pathlib import Path

# ─── EDIT THESE ───────────────────────────────────────────────
APP_ID = "YOUR_APP_ID"
SECRET = "YOUR_SECRET"
AUTH_TOKENS = [
    "TOKEN_1",
    # "TOKEN_2",
    # "TOKEN_3",
]
# ───────────────────────────────────────────────────────────────

config = {
    "app_id": APP_ID.strip(),
    "secret": SECRET.strip(),
    "auth_tokens": [t.strip() for t in AUTH_TOKENS if t.strip()],
    "download_dir": "/content/Qobuz",
    "quality": "hi-res-192",
    "folder_template": "{main_artist}/{year} - {album}",
    "track_template": "{track:02d} - {title}",
    "quality_fallback": True,
    "quality_fallback_path": ["hi-res-192", "hi-res", "cd"],
    "duration_check": True,
    "save_cover": True,
    "embed_metadata": True,
    "force_main_album_artist": True,
    "skip_existing": True,
    "retries": 3,
    "multi_disc": True,
    "on_final_failure": "delete_partial",
    "include_version": True,
    "strip_feat_from_album_title": False,
    "strip_feat_from_track_title": False,
}

# CLI always reads from here (root home in Colab)
cfg_dir = Path("/root/.config/qobuz-dl")
cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / "config.json"
cfg_path.write_text(json.dumps(config, indent=2))

print("Config saved →", cfg_path)
print("app_id:", config["app_id"])
print("tokens:", len(config["auth_tokens"]))
if not config["auth_tokens"] or config["app_id"].startswith("YOUR_"):
    print("⚠️  Replace APP_ID / SECRET / AUTH_TOKENS with your real values, then re-run this cell.")

## 3. Quick auth test (optional)

Should print `Status: 200` and some user info. If `401`, token or app_id/secret is wrong.

In [ ]:
import json, requests
from pathlib import Path

cfg = json.loads(Path("/root/.config/qobuz-dl/config.json").read_text())
r = requests.get(
    "https://www.qobuz.com/api.json/0.2/user/get",
    headers={
        "X-App-Id": cfg["app_id"],
        "X-User-Auth-Token": cfg["auth_tokens"][0],
    },
    timeout=20,
)
print("Status:", r.status_code)
print(r.text[:400])

## 4. Download

Use `play.qobuz.com` / `open.qobuz.com` URLs, or `ar-id` / `al-id` / `tr-id`.

Website URLs like `www.qobuz.com/us-en/interpreter/...` are **not** supported — use the numeric ID instead.

In [ ]:
%cd /content/qobuzdl-collab

# Examples (uncomment / edit one):

# Full artist discography:
!qobuz-dl dl ar-id 687008

# Single album:
# !qobuz-dl dl al-id r0mco98f1ytfa

# By URL:
# !qobuz-dl dl "https://open.qobuz.com/album/XXXX"
# !qobuz-dl dl "https://open.qobuz.com/artist/XXXX"

# Lower quality if many 401s on hi-res-192:
# !qobuz-dl dl ar-id 687008 -q hi-res

## 5. Zip & download to your computer

In [ ]:
!zip -r /content/Qobuz_Downloads.zip /content/Qobuz
from google.colab import files
files.download("/content/Qobuz_Downloads.zip")